# Causal Discovery Analysis

## Introduction

In [1]:
# Start the JVM and load tetrad-current.jar, then import all required packages.
import os
import sys
import json

import pandas as pd
import numpy as np

# Add repo root to path so we can import api
_repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

from api import tetrad, data, score, bootstrapping, algorithm, graph, knowledge, translate
tetrad.start("resources/tetrad-current.jar")

from scipy.stats import rankdata, norm
import shutil

py_output_dir = "pykumu_outputs"
os.makedirs(py_output_dir, exist_ok=True)


This notebook performs the necessary data transformations to the final table generated by [issue_social_smell_showcase.Rmd](https://github.com/sailuh/kaiaulu/blob/master/vignettes/issue_social_smell_showcase.Rmd) Notebook in order to perform Causal Analysis using Pykumu and Tetrad via JPype.


In [2]:
# dt = pd.read_csv("~/causal_tse/causal_modelling/1_openssl_social_smells_timeline.csv")
dt = pd.read_csv("resources/null_variable_dt.csv")

## Feature Engineering

### Formatting Data Types

In order to be loaded in Tetrad, some variables must be transformed from String to Integer due to data type limitations. 

#### CVE Data Type

We concatenate the last two digits of the year with the last four digits of the cve_id and convert into an integer. (E.g. 2006 and CVE ID XXX4339 becomes 06339).


In [3]:
if 'cve_id' in dt.columns and dt['cve_id'].dtype == object:
    last_two_digits_year = dt['cve_id'].str[6:8]
    last_four_digits_cve = dt['cve_id'].str[9:14]
    dt['cve_id'] = (last_two_digits_year + last_four_digits_cve).astype(int)
    print("Applied cve_id transformation.")
else:
    print("Skipped cve_id transformation (column missing or already numeric).")

Skipped cve_id transformation (column missing or already numeric).


Second, commit interval is transformed into `activity_0` and `activity_2` if the commit hash is missing or available respectively:

In [4]:
if 'commit_interval' in dt.columns:
    dt['activity_0'] = (dt['commit_interval'] == "").astype(int)
    dt['activity_2'] = (dt['commit_interval'] != "").astype(int)
    print("Applied activity_0/activity_2 transformation.")
else:
    print("Skipped activity transformation (commit_interval column missing).")

Skipped activity transformation (commit_interval column missing).


## Feature Renaming

A number of feature names are also shortened, so their visual representation do not take too much screen space:


In [5]:
rename_map = {
    "start_datetime": "start",
    "missing_links": "mis_link",
    "radio_silence": "silence",
    "code_only_devs": "code_dev",
    "code_files": "file",
    "ml_only_devs": "mail_dev",
    "ml_threads": "thread",
    "n_commits": "commit"
}

cols_to_rename = {k: v for k, v in rename_map.items() if k in dt.columns}
if cols_to_rename:
    dt = dt.rename(columns=cols_to_rename)
    expected_cols = ["cve_id", "activity_0", "activity_2", "start", "org_silo",
                     "mis_link", "silence", "code_dev", "file", "mail_dev",
                     "thread", "commit", "churn"]
    available_cols = [c for c in expected_cols if c in dt.columns]
    dt = dt[available_cols]
    print(f"Applied feature renaming: {list(cols_to_rename.keys())}")
    print(f"Selected columns: {available_cols}")
else:
    print("Skipped feature renaming (columns already renamed or missing).")

dt.to_csv(os.path.join(py_output_dir, "feature_renaming.csv"), index=False)


Skipped feature renaming (columns already renamed or missing).


### Missing Data Handling

We decided to remove rows from the dataset for which the mailing list data source is missing (i.e. 2000-2001).

In [6]:
if 'start' in dt.columns and dt['start'].dtype == object:
    dt['start'] = pd.to_datetime(dt['start'])
    dt = dt[(dt['start'].dt.year < 2000) | (dt['start'].dt.year > 2001)]
    print("Applied date filtering (removed 2000-2001 rows).")
else:
    print("Skipped date filtering (start is not a date string).")

Skipped date filtering (start is not a date string).


With respect to data missing due to inactivity during a given time period, any measures of features (counts) related to commits should all be 0.

In [7]:
if dt.isna().any().any():
    dt = dt.fillna(0)
    print("Applied fillna(0) â€” missing values found and filled.")
else:
    print("Skipped fillna (no missing values found).")

dt.to_csv(os.path.join(py_output_dir, "missing_data_handling.csv"), index=False)


Skipped fillna (no missing values found).


#### Convert "start" to Unix Timestamp

To use start in causal analysis, we convert it to a unix timestamp. 

In [8]:
if 'start' in dt.columns and pd.api.types.is_datetime64_any_dtype(dt['start']):
    dt['start'] = dt['start'].astype(np.int64) // 10**9
    print("Converted start to Unix timestamp.")
else:
    print("Skipped start conversion (already numeric or missing).")

dt.to_csv(os.path.join(py_output_dir, "feature_engineering.csv"), index=False)


Skipped start conversion (already numeric or missing).


### 1-Time Lag Features

In [9]:
lag_cols = ["org_silo", "mis_link", "silence", "code_dev", "file",
            "mail_dev", "thread", "commit", "churn"]

def add_time_lag(cve_table):
    if len(cve_table) < 2:
        for col in lag_cols:
            cve_table[col + "2"] = np.nan
        return cve_table
    else:
        current = cve_table.iloc[:-1].reset_index(drop=True)
        future = cve_table[lag_cols].iloc[1:].reset_index(drop=True)
        future.columns = [col + "2" for col in lag_cols]
        return pd.concat([current, future], axis=1)

In [10]:
if 'cve_id' in dt.columns and not any(c.endswith('2') for c in dt.columns if c not in ['activity_2']):
    lag_dt = (dt.sort_values(["cve_id", "start"])
                .groupby("cve_id", group_keys=False)
                .apply(add_time_lag)
                .reset_index(drop=True))
    print("Applied time lag features.")
else:
    lag_dt = dt.copy()
    print("Skipped time lag (lag columns already present or cve_id missing).")

lag_dt.to_csv(os.path.join(py_output_dir, "time_lag_features.csv"), index=False)


Skipped time lag (lag columns already present or cve_id missing).


### Remove Short CVEs 

We deleted CVEs (their associated rows) with 7 or fewer time periods.

In [11]:
if 'cve_id' in lag_dt.columns:
    short_cves = (lag_dt.groupby("cve_id").size()
                  .reset_index(name="n_rows")
                  .sort_values("n_rows")
                  .query("n_rows <= 7"))
    print("Identified short CVEs:")
    print(short_cves)
else:
    short_cves = pd.DataFrame(columns=["cve_id", "n_rows"])
    print("Skipped short CVE detection (cve_id column missing).")

short_cves.to_csv(os.path.join(py_output_dir, "short_cves.csv"), index=False)


Skipped short CVE detection (cve_id column missing).


In [12]:
if 'cve_id' in lag_dt.columns and len(short_cves) > 0:
    short_cve_ids = short_cves['cve_id'].values
    lag_dt = lag_dt[~lag_dt['cve_id'].isin(short_cve_ids)]
    print(f"Removed {len(short_cve_ids)} short CVEs.")
else:
    print("Skipped short CVE removal (no short CVEs or cve_id missing).")

Skipped short CVE removal (no short CVEs or cve_id missing).


## Addressing Determinism and High Intercorrelation Among Features

In [13]:
cor_cols = ["org_silo", "mis_link", "silence", "code_dev", "file",
            "mail_dev", "thread", "commit", "churn",
            "org_silo2", "mis_link2", "silence2", "code_dev2", "file2",
            "mail_dev2", "thread2", "commit2", "churn2"]

available_cor_cols = [c for c in cor_cols if c in lag_dt.columns]
if available_cor_cols:
    cor_table = lag_dt[available_cor_cols]
    cor_table.corr()
else:
    print("Skipped correlation analysis (columns not present).")

cor_matrix = lag_dt[available_cor_cols].corr()
cor_matrix.insert(0, "rn", cor_matrix.index)
cor_matrix.to_csv(os.path.join(py_output_dir, "correlation_matrix.csv"), index=False)


Due to high correlation, we perform 6 feature deletions (activity_0, activity_2, org_silo, org_silo2):

In [14]:
keep_cols = ["cve_id", "start", "mis_link", "silence", "code_dev", "file",
             "mail_dev", "thread", "commit", "churn",
             "mis_link2", "silence2", "code_dev2", "file2",
             "mail_dev2", "thread2", "commit2", "churn2"]

available_keep = [c for c in keep_cols if c in lag_dt.columns]
if 'org_silo' in lag_dt.columns or 'activity_0' in lag_dt.columns:
    lag_dt = lag_dt[available_keep]
    print("Applied feature deletion (removed correlated columns).")
else:
    print("Skipped feature deletion (columns already removed).")

Skipped feature deletion (columns already removed).


## Non-Parametric Transformation

We then apply the non-paranormal distribution transformation to numerical variables, to reduce the risk of violating the normal distribution when drawing causal conclusions.

In [15]:
def npn(data):
    """Non-paranormal (nonparanormal) transformation via rank-based Gaussian copula."""
    n = len(data)
    result = data.copy()
    for col in data.columns:
        ranks = rankdata(data[col])
        result[col] = norm.ppf(ranks / (n + 1))
    return result

if 'cve_id' in lag_dt.columns:
    lag_dt = pd.concat([lag_dt[["cve_id", "start"]],
                        npn(lag_dt.iloc[:, 2:])], axis=1)
    print("Applied non-paranormal transformation.")
    lag_dt.head()
else:
    print("Skipped non-paranormal transformation (data already transformed).")
    lag_dt.head()

lag_dt.to_csv(os.path.join(py_output_dir, "npn_transformed.csv"), index=False)


Skipped non-paranormal transformation (data already transformed).


### Binarized CVE Indicators

To represent the CVE Ids, we utilize indicator features. For every CVE ID, a new column is added to the table which can take values 0 or 1. The value is 1 if the row is associated to that CVE ID, or 0 otherwise.

In [16]:
if 'cve_id' in lag_dt.columns:
    binarize_cve_id = pd.DataFrame({
        "id": range(len(lag_dt)),
        "cve_id": "b_" + lag_dt["cve_id"].astype(str),
        "binary_value": 1
    })
    binarize_cve_id = binarize_cve_id.pivot(index="id", columns="cve_id",
                                            values="binary_value").fillna(0).astype(int)
    binarize_cve_id = binarize_cve_id.reset_index()
    pd.concat([lag_dt[["cve_id"]].reset_index(drop=True), binarize_cve_id], axis=1).head()
    print("Applied CVE binarization.")
else:
    binarize_cve_id = None
    print("Skipped CVE binarization (cve_id column missing â€” data already binarized).")

if binarize_cve_id is not None:
    pd.concat([lag_dt[["cve_id"]].reset_index(drop=True), binarize_cve_id], axis=1).to_csv(os.path.join(py_output_dir, "binarized_cve_indicators.csv"), index=False)


Skipped CVE binarization (cve_id column missing â€” data already binarized).


We can then remove the `cve_id` column, as the binary features represent the same information, and add the remaining columns to the analysis table:

In [17]:
if binarize_cve_id is not None:
    lag_dt = lag_dt[["start", "mis_link", "silence", "code_dev", "file",
                    "mail_dev", "thread", "commit", "churn",
                    "mis_link2", "silence2", "code_dev2", "file2",
                    "mail_dev2", "thread2", "commit2", "churn2"]].reset_index(drop=True)
    binarized_lag_dt = pd.concat([lag_dt, binarize_cve_id.drop(columns="id")], axis=1)
    print("Applied cve_id removal and binary column concatenation.")
else:
    # Data already has binarized columns â€” extract non-nv columns as binarized_lag_dt
    nv_cols = [c for c in lag_dt.columns if c.startswith("nv-")]
    binarized_lag_dt = lag_dt.drop(columns=nv_cols, errors='ignore')
    print("Skipped â€” using existing binarized columns from loaded data.")

binarized_lag_dt.to_csv(os.path.join(py_output_dir, "binarized_lag_dt.csv"), index=False)


Skipped â€” using existing binarized columns from loaded data.


### Add Null Features

An example of the randomization only showing the silence and nv-silence is shown below. In practice, for every column in `lag_dt` up to this point, we generated a replica column prefixed by `nv-`, including the binary features (which are then prefixed as `nv-b_`), but the replica columns have their values shuffled across the rows, hence the null (random) naming to them.


In [18]:
if not any(c.startswith("nv-") for c in lag_dt.columns):
    nv_lag_dt = binarized_lag_dt.copy()
    nv_lag_dt.columns = "nv-" + binarized_lag_dt.columns
    nv_lag_dt = nv_lag_dt.apply(lambda col: col.sample(frac=1).reset_index(drop=True))
    nv_lag_dt = pd.concat([binarized_lag_dt, nv_lag_dt], axis=1)
    print("Applied null variable features.")
    nv_lag_dt[["silence", "nv-silence"]].head()
else:
    nv_lag_dt = lag_dt.copy()
    print("Skipped null feature creation (nv- columns already present).")
    if "silence" in nv_lag_dt.columns and "nv-silence" in nv_lag_dt.columns:
        nv_lag_dt[["silence", "nv-silence"]].head()

Skipped null feature creation (nv- columns already present).


### Keep only 5 null indicator features

Introducing a null feature for all variables and features leads to too many features being introduced for causal search, causing heap memory errors in Tetrad. We preserve only a few of the nv binary indicator variables, as they lead to variable explosion and their pattern is easy to randomize. Position 138 includes all variables as null variables, plus five binary indicators as null variables. We consider this loss of null binary indicator features reasonable, as the randomization of a few blocks of values 1 or 0 will generally be equivalent. This in turn, allow us to perform more causal search runs, which we deem a fair trade-off. 

In [19]:
if nv_lag_dt.shape[1] > 138:
    nv_lag_dt = nv_lag_dt.iloc[:, :138]
    print(f"Trimmed to 138 columns.")
else:
    print(f"Skipped column trimming (already {nv_lag_dt.shape[1]} columns).")

# Convert all columns to float so Tetrad treats them as continuous (required for SEM BIC)
nv_lag_dt = nv_lag_dt.astype(float)
binarized_lag_dt = binarized_lag_dt.astype(float)
print(f"Converted nv_lag_dt and binarized_lag_dt to float.")

nv_lag_dt.to_csv(os.path.join(py_output_dir, "nv_lag_dt_final.csv"), index=False)


Skipped column trimming (already 138 columns).
Converted nv_lag_dt and binarized_lag_dt to float.


## FGES Null Variable Search

Since we are using JPype and TetradSearch directly (instead of causal-cmd), we do not need to save the dataset to CSV. Instead, we pass the pandas DataFrame directly to Tetrad.

We then set the output file configuration. 


In [20]:
# Output path configuration (for saving results later if needed)
output_folder_path = os.path.join(os.getcwd(), "null_search")

Finally, we perform causal search over our null dataset. In this Notebook we include both FGES and BOSS. To execute one or the other, modify `eval` to TRUE on either code block. This may take some time to execute. After the data is generated, the code block evaluation can be again set to FALSE, as the remaining analysis can be performed using the output file of either algorithms. 

The output `.json` file of Causal Search can also be loaded directly on Tetrad GUI for inspection.

This is the FGES search. Note in our local machine, we could not use the number of runs up to 1000, due to java memory heap errors.



In [21]:
# Set eval_fges = True to run, False to skip
eval_fges = True

if eval_fges:
    filename = "fges_bootstrap_null_search_500_runs_nv_binary_indicators"
    filepath = os.path.join(output_folder_path, filename + "_graph.json")

    state = data.load_continuous(nv_lag_dt)
    sem_bic = score.use_sem_bic(state["params"], penalty_discount=2, sem_bic_structure_prior=0, sem_bic_rule=1)
    bootstrapping.set_bootstrapping(state["params"], number_resampling=500,
                                    percent_resample_size=90, seed=32,
                                    add_original_dataset=True, resampling_with_replacement=True,
                                    resampling_ensemble=1)

    result = algorithm.run_fges(state["data"], state["params"], sem_bic, state["knowledge"],
                                symmetric_first_step=True, max_degree=1000,
                                faithfulness_assumed=True, parallelized=False)

    # Save graph as JSON
    fges_null_graph_json = graph.get_json(result["graph"])
    os.makedirs(output_folder_path, exist_ok=True)
    with open(filepath, 'w') as f:
        f.write(graph.convert_to_tetrad_gui_format(fges_null_graph_json))
    print(f"FGES null search graph saved to: {filepath}")

if eval_fges:
    shutil.copy2(filepath, os.path.join(py_output_dir, "null_search_graph.json"))


Bootstrap count = 1
Bootstrap count = 2
Bootstrap count = 3
Bootstrap count = 4
Bootstrap count = 5
Bootstrap count = 6
Bootstrap count = 7
Bootstrap count = 8
Bootstrap count = 9
Bootstrap count = 10
Bootstrap count = 11
Bootstrap count = 12
Bootstrap count = 13
Bootstrap count = 14
Bootstrap count = 15
Bootstrap count = 16
Bootstrap count = 17
Bootstrap count = 18
Bootstrap count = 19
Bootstrap count = 20
Bootstrap count = 21
Bootstrap count = 22
Bootstrap count = 23
Bootstrap count = 24
Bootstrap count = 25
Bootstrap count = 26
Bootstrap count = 27
Bootstrap count = 28
Bootstrap count = 29
Bootstrap count = 30
Bootstrap count = 31
Bootstrap count = 32
Bootstrap count = 33
Bootstrap count = 34
Bootstrap count = 35
Bootstrap count = 36
Bootstrap count = 37
Bootstrap count = 38
Bootstrap count = 39
Bootstrap count = 40
Bootstrap count = 41
Bootstrap count = 42
Bootstrap count = 43
Bootstrap count = 44
Bootstrap count = 45
Bootstrap count = 46
Bootstrap count = 47
Bootstrap count = 48
B

## BOSS Null Variable Search

This is the BOSS Search. We found BOSS scaled better, allowing us to increase the number of bootstraps up to 1000. Observe also the knowledge box is not specified at this point: We do not impose any restrictions when observing the formation of edges at random. Our final conclusions are derived from BOSS. Note also that we bootstrap on BOSS with 100% of the dataset, while in FGES we do so with 90% of the dataset. We use 100% in BOSS, because the algorithm already has random initialization. 


In [22]:
# Set eval_boss = True to run, False to skip
eval_boss = False

if eval_boss:
    filename = "boss_bootstrap_null_search_1000_runs_nv_binary_indicators"
    filepath = os.path.join(output_folder_path, filename + "_graph.json")

    state = data.load_continuous(nv_lag_dt)
    sem_bic = score.use_sem_bic(state["params"], penalty_discount=2, sem_bic_structure_prior=0, sem_bic_rule=1)
    bootstrapping.set_bootstrapping(state["params"], number_resampling=50,
                                    percent_resample_size=100, seed=32,
                                    add_original_dataset=True, resampling_with_replacement=True,
                                    resampling_ensemble=1)

    result = algorithm.run_boss(state["data"], state["params"], sem_bic, state["knowledge"],
                                num_starts=1, use_bes=False, time_lag=0,
                                use_data_order=True)

    # Save graph as JSON
    boss_null_graph_json = graph.get_json(result["graph"])
    os.makedirs(output_folder_path, exist_ok=True)
    with open(filepath, 'w') as f:
        f.write(graph.convert_to_tetrad_gui_format(boss_null_graph_json))
    print(f"BOSS null search graph saved to: {filepath}")

if eval_boss:
    shutil.copy2(filepath, os.path.join(py_output_dir, "null_search_graph.json"))


## Deriving the 1 PNEF Threshold

In our causal search above, we introduced null features over multiple bootstrap runs to observe how often our causal search form random edges (i.e. between our features and null features). We will use this information to derive a threshold, 1PNEF, we can use in our final causal search.


### Graph Examination

We now have our causal bootstrap graph as a .json file, which is output by Tetrad. Let's parse it into a tabular format to provide further intuition on how the 1 PNEF threshold is being determined. 

The nodes contain all our variables and null features In the off_chance a feature does not have any edge to it, this table allow us to still show it on the graph, as it would not appear on the "edge list" table.


In [23]:
parsed_graph = graph.parse_graph(filepath)
parsed_graph["nodes"].head()

parsed_graph["nodes"].to_csv(os.path.join(py_output_dir, "null_search_nodes.csv"), index=False)


Next is the edgeset table output by Tetrad. This table contains all the edges. Because we are performing multiple executions, each with a sample of the full dataset (as we are using a "bootstrap" approach), the probabilities represented here are the "ensemble" of all edges formed on each execution. In this Notebook, the preserved ensemble was used.


In [24]:
parsed_graph["edgeset"].head()

parsed_graph["edgeset"].to_csv(os.path.join(py_output_dir, "null_search_edgeset.csv"), index=False)


Lastly, we can examine the counts of each type of edge formed on each subgraph via the edge_type_probabilities table. Since the edgeset table probability already sums the probabilities from this table for every node pair, this information is presented here only for qualitative inspection, but it is not currently used in the subsequent steps.


In [25]:
parsed_graph["edge_type_probabilities"].head()

parsed_graph["edge_type_probabilities"].to_csv(os.path.join(py_output_dir, "null_search_edge_type_probabilities.csv"), index=False)


### Deriving 1 PNEF

As noted, our interest is to derive a threshold for the final causal search, using the information of this bootstrapped null feature causal search between the actual variables, and the random features. By this randome dge definition, our first step is to subset the `edgeset`table  to contain only the edge pairs that include null variables. A sample is shown below of the table where at least one of the two nodes is nv:


In [26]:
is_node1_nv = parsed_graph["edgeset"]["node1_name"].str.contains("nv-", regex=False)
is_node2_nv = parsed_graph["edgeset"]["node2_name"].str.contains("nv-", regex=False)
nv_edges = parsed_graph["edgeset"][is_node1_nv | is_node2_nv].copy()
nv_edges.head()

nv_edges.to_csv(os.path.join(py_output_dir, "nv_edges.csv"), index=False)


Next, we can derive a no_edge probability by subtracting 1 from the `probability` value. 


In [27]:
nv_edges["no_edge"] = 1 - nv_edges["probability"]
nv_edges.head()

,node1_name,node2_name,endpoint1,endpoint2,bold,highlighted,properties,probability,no_edge
36,b_173733,nv-b_102939,TAIL,ARROW,False,False,pd;pl,0.556886,0.443114


Our goal then is to identify the first percentile value of the no edge probability, i.e. the 1st percentile NoEdge Frequency value (1PNEF):


In [28]:
pnef_1 = float(nv_edges["no_edge"].quantile(0.01))
pnef_1

with open(os.path.join(py_output_dir, "pnef_1.txt"), "w") as f:
    f.write(str(pnef_1) + "\n")


What this threshold tell us is that, if executed 1000 runs, then the first percentile of all random edges formed had approximately 65% no formation of causal link. Another way to state this is that given entirely random variables, causal links were formed between them up to 35% of the time. In our final search, we then only keep causal links that, over 1000 runs, formed **more** than 35% of the time, under the assumption any causal link established less than that may be due to random chance. 

With the threshold defined, we can now proceed to the final causal search, which does not include null features. In this non null feature causal search, we also specify domain knowledge. 

## Non-Null Causal Search

### Domain Knowledge Causal Search without Null Variables

Domain knowledge is used to prohibit causal links to form among features. Here, we only defined temporal causal link restrictions. I.e. it does not make sense for features at 1-time-lag (future) to cause features on the present time.


In [29]:
#knowledge_file_path = os.path.expanduser("~/Downloads/knowledge_2.txt")
knowledge_file_path = ("resources/knowledge_box.txt")  

### Causal Search

As before, we specify the graph output file. 

In [30]:
output_folder_path = os.path.join(os.getcwd(), "domain_binarized_search")

We also have the choice of using FGES or BOSS here. In our final analysis, we used BOSS.

FGES Causal Search:

In [31]:
# Set eval_fges_domain = True to run, False to skip
eval_fges_domain = False

if eval_fges_domain:
    filename = "fges_bootstrap_binarized_search_500_runs_binary_indicators"
    filepath = os.path.join(output_folder_path, filename + "_graph.json")

    state = data.load_continuous(binarized_lag_dt)
    state["knowledge"] = knowledge.load_knowledge(knowledge_file_path)
    sem_bic = score.use_sem_bic(state["params"], penalty_discount=2, sem_bic_structure_prior=0, sem_bic_rule=1)
    bootstrapping.set_bootstrapping(state["params"], number_resampling=500,
                                    percent_resample_size=90, seed=32,
                                    add_original_dataset=True, resampling_with_replacement=True,
                                    resampling_ensemble=1)

    result = algorithm.run_fges(state["data"], state["params"], sem_bic, state["knowledge"],
                                symmetric_first_step=True, max_degree=1000,
                                faithfulness_assumed=True, parallelized=False)

    # Save graph as JSON
    fges_domain_graph_json = graph.get_json(result["graph"])
    os.makedirs(output_folder_path, exist_ok=True)
    with open(filepath, 'w') as f:
        f.write(graph.convert_to_tetrad_gui_format(fges_domain_graph_json))
    print(f"FGES domain search graph saved to: {filepath}")

if eval_fges_domain:
    shutil.copy2(filepath, os.path.join(py_output_dir, "domain_search_graph.json"))


BOSS Causal Search:


In [32]:
# Set eval_boss_domain = True to run, False to skip
eval_boss_domain = True

if eval_boss_domain:
    filename = "boss_bootstrap_binarized_search_1000_runs_binary_indicators"
    filepath = os.path.join(output_folder_path, filename + "_graph.json")

    state = data.load_continuous(binarized_lag_dt)
    state["knowledge"] = knowledge.load_knowledge(knowledge_file_path)
    sem_bic = score.use_sem_bic(state["params"], penalty_discount=2, sem_bic_structure_prior=0, sem_bic_rule=1)
    bootstrapping.set_bootstrapping(state["params"], number_resampling=600,
                                    percent_resample_size=100, seed=32,
                                    add_original_dataset=True, resampling_with_replacement=True,
                                    resampling_ensemble=1)

    from edu.cmu.tetrad.util import Params
    state["params"].set(Params.NUM_THREADS, 15)

    result = algorithm.run_boss(state["data"], state["params"], sem_bic, state["knowledge"],
                                num_starts=1, use_bes=False, time_lag=0,
                                use_data_order=True)

    # Save graph as JSON
    boss_domain_graph_json = graph.get_json(result["graph"])
    os.makedirs(output_folder_path, exist_ok=True)
    with open(filepath, 'w') as f:
        f.write(graph.convert_to_tetrad_gui_format(boss_domain_graph_json))
    print(f"BOSS domain search graph saved to: {filepath}")

if eval_boss_domain:
    shutil.copy2(filepath, os.path.join(py_output_dir, "domain_search_graph.json"))



Loading knowledge.
Adding to tier 1 start
Adding to tier 1 mis_link
Adding to tier 1 silence
Adding to tier 1 code_dev
Adding to tier 1 file
Adding to tier 1 mail_dev
Adding to tier 1 thread
Adding to tier 1 commit
Adding to tier 1 churn
Adding to tier 2 mis_link2
Adding to tier 2 silence2
Adding to tier 2 code_dev2
Adding to tier 2 file2
Adding to tier 2 mail_dev2
Adding to tier 2 thread2
Adding to tier 2 commit2
Adding to tier 2 churn2
Bootstrap count = 1
Bootstrap count = 2
Bootstrap count = 3
Bootstrap count = 4
Bootstrap count = 5
Bootstrap count = 6
Bootstrap count = 7
Bootstrap count = 8
Bootstrap count = 9
Bootstrap count = 10
Bootstrap count = 11
Bootstrap count = 12
Bootstrap count = 13
Bootstrap count = 14
Bootstrap count = 15
Bootstrap count = 16
Bootstrap count = 17
Bootstrap count = 18
Bootstrap count = 19
Bootstrap count = 20
Bootstrap count = 21
Bootstrap count = 22
Bootstrap count = 23
Bootstrap count = 24
Bootstrap count = 25
Bootstrap count = 26
Bootstrap count = 27

## Applying 1PNEF Threshold

We load the final causal search, and then apply the 1PNEF threshold derived from the prior causal search here. A sample of the causal graph nodes and edges is shown below:


In [33]:
parsed_graph = graph.parse_graph(filepath)
print("Nodes:")
print(parsed_graph["nodes"].head())
print("\nEdgeset:")
print(parsed_graph["edgeset"].head())
print("\nEdge Type Probabilities:")
print(parsed_graph["edge_type_probabilities"].head())

parsed_graph["nodes"].to_csv(os.path.join(py_output_dir, "final_search_nodes.csv"), index=False)
parsed_graph["edgeset"].to_csv(os.path.join(py_output_dir, "final_search_edgeset.csv"), index=False)
parsed_graph["edge_type_probabilities"].to_csv(os.path.join(py_output_dir, "final_search_edge_type_probabilities.csv"), index=False)


Nodes:
  node_name
0  b_100433
1  b_100740
2  b_100742
3  b_102939
4  b_103864

Edgeset:
  node1_name node2_name endpoint1 endpoint2   bold  highlighted properties  \
0      start   b_160705      TAIL     ARROW  False        False        NaN   
1    silence   mail_dev      TAIL     ARROW  False        False        NaN   
2     commit  code_dev2      TAIL     ARROW  False        False      dd;nl   
3    silence   silence2      TAIL     ARROW  False        False      dd;nl   
4      start    commit2      TAIL     ARROW  False        False      dd;nl   

   probability  
0     0.712146  
1     1.000000  
2     0.998336  
3     0.985025  
4     0.903494  

Edge Type Probabilities:
  node1_name node2_name edge_type properties  probability
0      start   b_160705        ta        NaN     0.522463
1      start   b_160705       nil        NaN     0.287854
2      start   b_160705        at        NaN     0.183028
3      start   b_160705        tt        NaN     0.006656
4    silence   mail_dev 

### Applying 1PNEF Threshold

Edges which may have been formed at random are filtered here:

In [34]:
edges = parsed_graph["edgeset"].copy()
edges["no_edge"] = 1 - edges["probability"]
edges_1pnef = edges[edges["no_edge"] <= pnef_1].copy()
edges_1pnef

edges_1pnef.to_csv(os.path.join(py_output_dir, "edges_1pnef.csv"), index=False)


## Results 

With the final causal graph trimmed, we can now inspect it to draw conclusions from it. Causal graphs may form cycles, and also have undirected edges. We define a function to check for cycles here and use it below for inspection.


### Full Causal Graph 1-PNEF Trimmed 

First, we can inspect the full causal graph.

### Sub-Graphs of Effort Variables and Parents
